Title: Autonomous_SQL_Agent_Supply_Chain_Risk

Objective: "Automating supply chain risk detection using Gemini 3 and Relational Data Modeling."

Tech Stack: Python, LangChain, SQLite, Google Gemini.

Dataset: Link to the Kaggle dataset you used.

In [37]:
!pip install langchain langchain-community langchain-openai pandas pysqlite3-binary langchain-experimental langchain langchain-core langchain-community langchain-experimental

In [38]:
import pandas as pd
import sqlite3

# Load the Kaggle dataset
df = pd.read_csv('data.csv')

# Create a local SQL database
conn = sqlite3.connect('supply_chain.db')
df.to_sql('orders', conn, index=False, if_exists='replace')

print("Database created with columns:", df.columns.tolist())

Database created with columns: ['Order_ID', 'Buyer_ID', 'Supplier_ID', 'Product_Category', 'Quantity_Ordered', 'Order_Date', 'Dispatch_Date', 'Delivery_Date', 'Shipping_Mode', 'Order_Value_USD', 'Delay_Days', 'Disruption_Type', 'Disruption_Severity', 'Historical_Disruption_Count', 'Supplier_Reliability_Score', 'Organization_ID', 'Dominant_Buyer_Flag', 'Available_Historical_Records', 'Data_Sharing_Consent', 'Federated_Round', 'Parameter_Change_Magnitude', 'Communication_Cost_MB', 'Energy_Consumption_Joules', 'Supply_Risk_Flag']


In [39]:
from langchain_community.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain
from langchain_openai import ChatOpenAI

# Connect to your DB
db = SQLDatabase.from_uri("sqlite:///supply_chain.db")

In [40]:
!pip install -U langchain-google-genai

In [41]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain
from langchain_core.prompts import PromptTemplate

# 1. Setup API Key
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

# 2. Initialize the latest Gemini 3 Flash model
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

# 3. Connect to Database
db = SQLDatabase.from_uri("sqlite:///supply_chain.db")

# 4. Professional Persona Prompt with explicit schema and no markdown rule
template = """You are a Senior Supply Chain Risk Analyst.
Given an input question, first create a syntactically correct SQLite query to run against the `orders` table.

The `orders` table has the following columns: ['Order_ID', 'Buyer_ID', 'Supplier_ID', 'Product_Category', 'Quantity_Ordered', 'Order_Date', 'Dispatch_Date', 'Delivery_Date', 'Shipping_Mode', 'Order_Value_USD', 'Delay_Days', 'Disruption_Type', 'Disruption_Severity', 'Historical_Disruption_Count', 'Supplier_Reliability_Score', 'Organization_ID', 'Dominant_Buyer_Flag', 'Available_Historical_Records', 'Data_Sharing_Consent', 'Federated_Round', 'Parameter_Change_Magnitude', 'Communication_Cost_MB', 'Energy_Consumption_Joules', 'Supply_Risk_Flag'].

IMPORTANT: The SQLQuery must be RAW TEXT ONLY. Do NOT use markdown, do NOT use ```sql, and do NOT use bold.

Use the following format:
Question: {input}
SQLQuery: SELECT ...
SQLResult: Result of the SQLQuery
Answer: Final response with a business mitigation strategy based on the data.

Question: {input}"""

prompt = PromptTemplate(template=template, input_variables=["input"])

# 5. Initialize the Chain
db_chain = SQLDatabaseChain.from_llm(llm, db, prompt=prompt, verbose=True)

print("✅ System Updated to Gemini 3 with improved prompt! You are ready to go.")

✅ System Updated to Gemini 3 with improved prompt! You are ready to go.


In [42]:
db_chain.invoke("Identify the top 3 suppliers with the lowest 'Supplier_Reliability_Score' and highest 'Order_Value_USD'.")



> Entering new SQLDatabaseChain chain...
Identify the top 3 suppliers with the lowest 'Supplier_Reliability_Score' and highest 'Order_Value_USD'.
SQLQuery:SELECT Supplier_ID, Supplier_Reliability_Score, Order_Value_USD FROM orders ORDER BY Supplier_Reliability_Score ASC, Order_Value_USD DESC LIMIT 3;
SQLResult: [('S20', 0.5, 40650.63), ('S9', 0.5, 38695.53), ('S17', 0.5, 38394.56)]
Answer:SQLQuery: SELECT Supplier_ID, Supplier_Reliability_Score, Order_Value_USD FROM orders ORDER BY Supplier_Reliability_Score ASC, Order_Value_USD DESC LIMIT 3;
> Finished chain.


{'query': "Identify the top 3 suppliers with the lowest 'Supplier_Reliability_Score' and highest 'Order_Value_USD'.",
 'result': 'SQLQuery: SELECT Supplier_ID, Supplier_Reliability_Score, Order_Value_USD FROM orders ORDER BY Supplier_Reliability_Score ASC, Order_Value_USD DESC LIMIT 3;'}

In [43]:
db_chain.invoke("Identify any product categories where more than 40% of the total order value is tied to suppliers with a reliability score below 60. Rank these categories by 'Total Dollar Exposure")



> Entering new SQLDatabaseChain chain...
Identify any product categories where more than 40% of the total order value is tied to suppliers with a reliability score below 60. Rank these categories by 'Total Dollar Exposure
SQLQuery:SELECT Product_Category, SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0 END) AS Total_Dollar_Exposure FROM orders GROUP BY Product_Category HAVING (SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0 END) * 1.0 / SUM(Order_Value_USD)) > 0.40 ORDER BY Total_Dollar_Exposure DESC
SQLResult: [('Electronics', 5688988.499999997), ('Textiles', 5234273.249999999), ('Machinery', 5034029.400000002), ('Food', 4709296.130000002), ('Pharma', 4623486.390000001)]
Answer:SQLQuery: SELECT Product_Category, SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0 END) AS Total_Dollar_Exposure FROM orders GROUP BY Product_Category HAVING (SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0

{'query': "Identify any product categories where more than 40% of the total order value is tied to suppliers with a reliability score below 60. Rank these categories by 'Total Dollar Exposure",
 'result': 'SQLQuery: SELECT Product_Category, SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0 END) AS Total_Dollar_Exposure FROM orders GROUP BY Product_Category HAVING (SUM(CASE WHEN Supplier_Reliability_Score < 60 THEN Order_Value_USD ELSE 0 END) * 1.0 / SUM(Order_Value_USD)) > 0.40 ORDER BY Total_Dollar_Exposure DESC'}